In [1]:
using LowLevelFEM

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
structured_rect_mesh()

mat = Material("body")
PT = Problem([mat], type=:ScalarField, dim=2, field=:T, rhs_field=:q)

Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q)

In [3]:
k = mat.k
ρ = mat.ρ
c = mat.c

4.2e8

In [4]:
K1 = ∫(Grad(PT) ⋅ k ⋅ Grad(PT))
C1 = ∫(PT ⋅ PT * (ρ * c))

sparse([1, 5, 40, 41, 2, 13, 14, 113, 3, 22  …  121, 3, 21, 22, 23, 24, 111, 112, 120, 121], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  120, 121, 121, 121, 121, 121, 121, 121, 121, 121], [0.0036633333333333314, 0.0018316666666666663, 0.0018316666666666663, 0.0009158333333333332, 0.0036633333333333322, 0.0018316666666666666, 0.0018316666666666666, 0.0009158333333333332, 0.0036633333333333305, 0.0018316666666666663  …  0.003663333333333332, 0.0009158333333333332, 0.0009158333333333336, 0.0036633333333333327, 0.0036633333333333327, 0.0009158333333333332, 0.0009158333333333325, 0.0036633333333333322, 0.003663333333333332, 0.014653333333333326], 121, 121)

In [5]:
q1 = ScalarField[]
for i in 1:1
    g = ScalarField(PT, "right", (x, y, z) -> 100 - i)
    q0 = ∫(PT ⋅ g, Γ="right")
    push!(q1, q0)
end
q1 = mergeFields(q1)

nodal ScalarField
[0.0; 4.95; … ; 0.0; 0.0;;]

In [6]:
h = ScalarField(PT, "left", (x, y, z, t) -> t / 1000, steps=100)
bc = BoundaryCondition("left", problem=PT, T=h)

BoundaryCondition("left", Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :T, :q), Dict{Symbol, Union{Function, Number, ScalarField}}(:T => ScalarField([[0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099], [0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099], [0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099], [0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099], [0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099], [0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099], [0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099], [0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099], [0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099], [0.0 0.001 … 0.098 0.099; 0.0 0.001 … 0.098 0.099]], Matrix{Float64}(undef, 0, 0), [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0  …  90.

In [7]:
λmax = largestEigenValue(K1, C1)
dt = 2 / λmax

6.105555555555531e-5

In [8]:
T0 = ScalarField(PT, "body", 0)
#T0 = ScalarField(T0, steps=100)
T0 = elementsToNodes(T0)

nodal ScalarField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [9]:
T = FDM(K1, C1, q1, T0, 100, dt, support=[bc])

nodal ScalarField
[0.0 0.001 … 0.098 0.099; 0.0 0.05185449733122283 … 0.7080189157194784 0.7116631279780912; … ; 0.0 -0.008896825327002342 … 0.5097315835108726 0.5132740585959633; 0.0 -0.00889682532700234 … 0.5097315835108727 0.5132740585959629]

In [10]:
showDoFResults(q1, name="q")
showDoFResults(T, name="T", visible=true)

1

In [11]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
